In [1]:
import sys, time
from pathlib import Path
from collections import Counter
import polars as pl

sys.path.append('..')
from src.config import load_config
cfg = load_config()

PROC = Path('..') / 'data' / 'processed'
FILES = {
    'depth0': PROC / 'depth0.parquet',
    'depth1': PROC / 'depth1_agg.parquet',
    'depth2': PROC / 'depth2_agg.parquet',
}
print('polars', pl.__version__)
print('config:', list(cfg.keys()) if isinstance(cfg, dict) else type(cfg))
for k, f in FILES.items():
    print(f'{k}: {f.stat().st_size/1e6:,.1f} MB')

polars 2.0.0
config: ['project', 'seed', 'paths', 'columns', 'split', 'champion', 'challenger', 'conformal', 'monitoring']
depth0: 186.3 MB
depth1: 394.9 MB
depth2: 57.5 MB


In [2]:
BYTES = {'Float64': 8, 'Int64': 8, 'UInt64': 8, 'Float32': 4, 'Int32': 4, 'UInt32': 4,
         'Date': 4, 'Int16': 2, 'UInt16': 2, 'Int8': 1, 'UInt8': 1, 'Boolean': 1}
SCHEMA, IDS = {}, {}

for k, f in FILES.items():
    lf = pl.scan_parquet(f)
    sch = lf.collect_schema()
    ids = lf.select('case_id').collect()['case_id']
    SCHEMA[k], IDS[k] = sch, ids
    n = ids.len()
    dtypes = [str(d) for c, d in sch.items() if c != 'case_id']
    n_str = sum(d == 'String' for d in dtypes)
    gb = sum(BYTES.get(d, 8) for d in dtypes if d != 'String') * n / 1e9
    print(f'\n== {k}: {n:,} dòng × {len(sch)} cột | case_id duy nhất: {ids.n_unique() == n}')
    print('  kiểu:', dict(Counter(dtypes)))
    print(f'  RAM ước tính (chưa tính {n_str} cột String): {gb:.2f} GB')

# Tập case_id có giống nhau giữa 3 file không
base_ids = IDS['depth0']
for k in ['depth1', 'depth2']:
    same = IDS[k].len() == base_ids.len() and IDS[k].is_in(base_ids).all()
    print(f'\n{k}: cùng tập case_id với depth0: {same}')

# Trùng tên cột giữa các file (ngoài case_id)
cols = {k: set(s.keys()) - {'case_id'} for k, s in SCHEMA.items()}
for a, b in [('depth0', 'depth1'), ('depth0', 'depth2'), ('depth1', 'depth2')]:
    print(f'trùng tên cột {a} ∩ {b}:', sorted(cols[a] & cols[b])[:10])

# Cột tổ chức ở depth0 (không mang hậu tố P/A/D/M/T/L của bảng gốc)
meta = [c for c in SCHEMA['depth0'] if c[-1] not in 'PADMTL']
print('\ndepth0 — cột không có hậu tố:', meta)


== depth0: 1,526,659 dòng × 225 cột | case_id duy nhất: True
  kiểu: {'Date': 1, 'Int64': 3, 'Float32': 81, 'Float64': 107, 'String': 25, 'Boolean': 6, 'Categorical': 1}
  RAM ước tính (chưa tính 25 cột String): 1.87 GB

== depth1: 1,526,659 dòng × 412 cột | case_id duy nhất: True
  kiểu: {'UInt32': 45, 'Float32': 339, 'String': 22, 'Boolean': 5}
  RAM ước tính (chưa tính 22 cột String): 2.35 GB

== depth2: 1,526,659 dòng × 106 cột | case_id duy nhất: True
  kiểu: {'UInt32': 75, 'Int64': 9, 'Float64': 21}
  RAM ước tính (chưa tính 0 cột String): 0.82 GB

depth1: cùng tập case_id với depth0: True

depth2: cùng tập case_id với depth0: True
trùng tên cột depth0 ∩ depth1: []
trùng tên cột depth0 ∩ depth2: []
trùng tên cột depth1 ∩ depth2: []

depth0 — cột không có hậu tố: ['case_id', 'date_decision', 'MONTH', 'target', 'split']


C:\Users\HOME\AppData\Local\Temp\ipykernel_17204\1564571485.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  same = IDS[k].len() == base_ids.len() and IDS[k].is_in(base_ids).all()


In [3]:
PH = 'a55475b1'
META = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target', 'split']
d0 = pl.scan_parquet(FILES['depth0'])
d1 = pl.scan_parquet(FILES['depth1'])
pl.Config(tbl_rows=60, tbl_cols=-1, tbl_width_chars=200, fmt_str_lengths=60)

# (a) Cột split có khớp với cách chia theo WEEK_NUM không
print('config split:', cfg['split'])
print(d0.group_by('split').agg(
    n=pl.len(), wk_min=pl.col('WEEK_NUM').min(), wk_max=pl.col('WEEK_NUM').max(),
    default_rate=pl.col('target').mean()).sort('wk_min').collect())

# (b) Cột D ở depth0: đã đổi sang số ngày chưa, có giá trị > 0 không
d_cols = [c for c in SCHEMA['depth0'] if c.endswith('D') and c not in META]
num_d = [c for c in d_cols if SCHEMA['depth0'][c].is_numeric()]
print(f'\n(b) {len(d_cols)} cột D; chưa ở dạng số:',
      {c: str(SCHEMA['depth0'][c]) for c in d_cols if c not in num_d})
r = d0.select([e for c in num_d for e in (
        pl.col(c).count().alias(f'{c}|n'),
        pl.col(c).median().alias(f'{c}|p50'),
        pl.col(c).max().alias(f'{c}|max'),
        (pl.col(c) > 0).sum().alias(f'{c}|n_pos'))]).collect().row(0, named=True)
tbl_d = pl.DataFrame([{'col': c, **{k: float(r[f'{c}|{k}'] or 0) for k in ['n', 'p50', 'max', 'n_pos']}}
                      for c in num_d])
print(f'{tbl_d.height} cột D dạng số, trong đó {tbl_d.filter(pl.col("n_pos") > 0).height} cột có giá trị > 0:')
print(tbl_d.filter(pl.col('n_pos') > 0).sort('n_pos', descending=True))

# (c) Placeholder a55475b1 trong các cột chuỗi của depth0 và depth1
def ph_report(lf, sch, name):
    sc = [c for c, d in sch.items() if d in (pl.String, pl.Categorical) and c not in META]
    r = lf.select([e for c in sc for e in (
            (pl.col(c).cast(pl.String) == PH).fill_null(False).mean().alias(f'{c}|ph'),
            pl.col(c).is_null().mean().alias(f'{c}|null'),
            pl.col(c).n_unique().alias(f'{c}|nu'))]).collect().row(0, named=True)
    t = pl.DataFrame([{'col': c, 'ph': r[f'{c}|ph'], 'null': r[f'{c}|null'], 'nu': r[f'{c}|nu']} for c in sc])
    print(f'\n(c) {name}: {len(sc)} cột chuỗi, {t.filter(pl.col("ph") > 0).height} cột chứa {PH}')
    print(t.sort('ph', descending=True))

ph_report(d0, SCHEMA['depth0'], 'depth0')
ph_report(d1, SCHEMA['depth1'], 'depth1')

# (d) n_unique của cột M ở depth1: có đếm placeholder không
nu_cols = [c for c in SCHEMA['depth1']
           if 'uniq' in c.split('__')[-1] and len(c.split('__')) >= 3 and c.split('__')[-2].endswith('M')]
r = d1.select([e for c in nu_cols for e in (
        pl.col(c).count().alias(f'{c}|n'),
        (pl.col(c) == 0).sum().alias(f'{c}|eq0'),
        (pl.col(c) == 1).sum().alias(f'{c}|eq1'),
        (pl.col(c) >= 2).sum().alias(f'{c}|ge2'))]).collect().row(0, named=True)
tbl_nu = pl.DataFrame([{'col': c, **{k: r[f'{c}|{k}'] for k in ['n', 'eq0', 'eq1', 'ge2']}} for c in nu_cols])
print(f'\n(d) depth1: {len(nu_cols)} cột n_unique của cột M')
print(tbl_nu.with_columns((pl.col('eq0') / pl.col('n')).alias('ty_le_0')).sort('ty_le_0'))

config split: {'train': [0, 43], 'valid': [44, 51], 'calib': [52, 55], 'oot1': [56, 61], 'oot2': [62, 91]}
shape: (5, 5)
┌───────┬────────┬────────┬────────┬──────────────┐
│ split ┆ n      ┆ wk_min ┆ wk_max ┆ default_rate │
│ ---   ┆ ---    ┆ ---    ┆ ---    ┆ ---          │
│ cat   ┆ u32    ┆ i64    ┆ i64    ┆ f64          │
╞═══════╪════════╪════════╪════════╪══════════════╡
│ train ┆ 841500 ┆ 0      ┆ 43     ┆ 0.029755     │
│ valid ┆ 224421 ┆ 44     ┆ 51     ┆ 0.035157     │
│ calib ┆ 82387  ┆ 52     ┆ 55     ┆ 0.038477     │
│ oot1  ┆ 102273 ┆ 56     ┆ 61     ┆ 0.048312     │
│ oot2  ┆ 276078 ┆ 62     ┆ 91     ┆ 0.025189     │
└───────┴────────┴────────┴────────┴──────────────┘

(b) 24 cột D; chưa ở dạng số: {}
24 cột D dạng số, trong đó 20 cột có giá trị > 0:
shape: (20, 5)
┌──────────────────────────────┬────────────┬─────────┬──────┬──────────┐
│ col                          ┆ n          ┆ p50     ┆ max  ┆ n_pos    │
│ ---                          ┆ ---        ┆ ---     ┆ --- 

In [4]:
D_DROP = ['responsedate_1012D', 'responsedate_4527233D', 'responsedate_4917613D',
          'assignmentdate_4527235D', 'lastrepayingdate_696D']
D_KEEP_FUTURE = ['firstdatedue_489D']
D_NULL_POS = [c for c in tbl_d.filter(pl.col('n_pos') > 0)['col'].to_list()
              if c not in D_DROP + D_KEEP_FUTURE]

# Mô tả cột từ feature_definitions.csv (nếu có trong data/raw)
FD = Path('..') / 'data' / 'raw' / 'feature_definitions.csv'
if FD.exists():
    fd = pl.read_csv(FD)
    print(fd.filter(pl.col(fd.columns[0]).is_in(D_DROP + D_KEEP_FUTURE + D_NULL_POS)))
else:
    print('Không thấy', FD)

# Default rate trên Train: nhóm > 0 (sẽ bị che) so với nhóm ≤ 0 và nhóm null gốc
tr = d0.filter(pl.col('split') == 'train')
rows = []
for c in D_NULL_POS + D_KEEP_FUTURE:
    r = tr.select(
        n_pos=(pl.col(c) > 0).sum(),
        n_null=pl.col(c).is_null().sum(),
        dr_pos=pl.col('target').filter(pl.col(c) > 0).mean(),
        dr_le0=pl.col('target').filter(pl.col(c) <= 0).mean(),
        dr_null=pl.col('target').filter(pl.col(c).is_null()).mean(),
    ).collect().row(0, named=True)
    rows.append({'col': c, **r})
print(pl.DataFrame(rows).with_columns(
    (pl.col('dr_pos') / pl.col('dr_null')).alias('pos_vs_null'),
    (pl.col('n_pos') / (pl.col('n_pos') + pl.col('n_null'))).alias('ty_le_pos_trong_null_moi'),
).sort('n_pos', descending=True))

Không thấy ..\data\raw\feature_definitions.csv
shape: (15, 8)
┌──────────────────────────────┬───────┬────────┬──────────┬──────────┬──────────┬─────────────┬──────────────────────────┐
│ col                          ┆ n_pos ┆ n_null ┆ dr_pos   ┆ dr_le0   ┆ dr_null  ┆ pos_vs_null ┆ ty_le_pos_trong_null_moi │
│ ---                          ┆ ---   ┆ ---    ┆ ---      ┆ ---      ┆ ---      ┆ ---         ┆ ---                      │
│ str                          ┆ i64   ┆ i64    ┆ f64      ┆ f64      ┆ f64      ┆ f64         ┆ f64                      │
╞══════════════════════════════╪═══════╪════════╪══════════╪══════════╪══════════╪═════════════╪══════════════════════════╡
│ lastapplicationdate_877D     ┆ 59203 ┆ 185126 ┆ 0.045369 ┆ 0.030544 ┆ 0.022217 ┆ 2.042072    ┆ 0.242309                 │
│ validfrom_1069D              ┆ 38620 ┆ 735054 ┆ 0.033092 ┆ 0.028573 ┆ 0.029689 ┆ 1.114611    ┆ 0.049918                 │
│ dtlastpmtallstes_4499206D    ┆ 35835 ┆ 688619 ┆ 0.036082 ┆ 0.035806 

In [5]:
# Cell 5 — Ghép depth0 + depth1 + depth2 → features.parquet
D_DROP0 = ['responsedate_1012D', 'responsedate_4527233D', 'responsedate_4917613D',
           'assignmentdate_4527235D', 'lastrepayingdate_696D',
           'lastapplicationdate_877D', 'lastrejectdate_50D',
           'datelastunpaid_3546854D', 'lastdelinqdate_224D']
D_KEEP_FUTURE0 = ['firstdatedue_489D']
D_NULL0 = [c for c in tbl_d.filter(pl.col('n_pos') > 0)['col'].to_list()
           if c not in D_DROP0 + D_KEEP_FUTURE0]
print('D_NULL0:', len(D_NULL0), D_NULL0)

def clean_strings(lf):
    sc = [c for c, d in lf.collect_schema().items() if d == pl.String and c not in META]
    return lf.with_columns([pl.when(pl.col(c) == PH).then(None).otherwise(pl.col(c)).alias(c) for c in sc])

def to_f32(lf):
    return lf.with_columns([pl.col(c).cast(pl.Float32)
                            for c, d in lf.collect_schema().items() if d == pl.Float64])

lf0 = (pl.scan_parquet(FILES['depth0'])
       .drop(D_DROP0)
       .with_columns([pl.when(pl.col(c) > 0).then(None).otherwise(pl.col(c)).alias(c) for c in D_NULL0]))
lf0 = to_f32(clean_strings(lf0))
lf1 = clean_strings(pl.scan_parquet(FILES['depth1']))
lf2 = to_f32(pl.scan_parquet(FILES['depth2']))

OUT_F = PROC / 'features.parquet'
t0 = time.time()
(lf0.join(lf1, on='case_id', how='left')
    .join(lf2, on='case_id', how='left')
    .sink_parquet(OUT_F))
print(f'Ghi xong {OUT_F.name}: {OUT_F.stat().st_size/1e6:,.1f} MB ({time.time()-t0:.0f}s)')

# Kiểm tra file kết quả
F = pl.scan_parquet(OUT_F)
sch = F.collect_schema()
ids = F.select('case_id').collect()['case_id']
str_cols = [c for c, d in sch.items() if d == pl.String and c not in META]
chk = F.select(
    *[(pl.col(c) == PH).sum().alias(c) for c in str_cols]
).collect().row(0, named=True)
mx = F.select([pl.col(c).max() for c in D_NULL0]).collect().row(0, named=True)

print(f'{ids.len():,} dòng × {len(sch)} cột | case_id duy nhất: {ids.n_unique() == ids.len()}')
print('kiểu:', dict(Counter(str(d) for c, d in sch.items() if c != 'case_id')))
print('còn cột DROP_D:', [c for c in D_DROP0 if c in sch])
print('cột chuỗi còn a55475b1:', {c: v for c, v in chk.items() if v})
print('max của D_NULL0 (phải ≤ 0 hoặc null):', {c: v for c, v in mx.items() if v is not None and v > 0})
print('cột tổ chức có đủ:', all(c in sch for c in META))

D_NULL0: 10 ['datefirstoffer_1144D', 'dtlastpmtallstes_4499206D', 'firstclxcampaign_1125D', 'lastactivateddate_801D', 'lastapprdate_640D', 'maxdpdinstldate_3546855D', 'payvacationpostpone_4187118D', 'validfrom_1069D', 'assignmentdate_238D', 'assignmentdate_4955616D']
Ghi xong features.parquet: 618.5 MB (24s)
1,526,659 dòng × 732 cột | case_id duy nhất: True
kiểu: {'Date': 1, 'Int64': 12, 'Float32': 539, 'String': 47, 'Boolean': 11, 'Categorical': 1, 'UInt32': 120}
còn cột DROP_D: []
cột chuỗi còn a55475b1: {}
max của D_NULL0 (phải ≤ 0 hoặc null): {}
cột tổ chức có đủ: True
